# Prepocessing the data

In [ ]:
# !pip install openvino==2024.3.0

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
cols="""duration,
protocol_type,
service,
flag,
src_bytes,
dst_bytes,
land,
wrong_fragment,
urgent,
hot,
num_failed_logins,
logged_in,
num_compromised,
root_shell,
su_attempted,
num_root,
num_file_creations,
num_shells,
num_access_files,
num_outbound_cmds,
is_host_login,
is_guest_login,
count,
srv_count,
serror_rate,
srv_serror_rate,
rerror_rate,
srv_rerror_rate,
same_srv_rate,
diff_srv_rate,
srv_diff_host_rate,
dst_host_count,
dst_host_srv_count,
dst_host_same_srv_rate,
dst_host_diff_srv_rate,
dst_host_same_src_port_rate,
dst_host_srv_diff_host_rate,
dst_host_serror_rate,
dst_host_srv_serror_rate,
dst_host_rerror_rate,
dst_host_srv_rerror_rate"""

columns=[]
for c in cols.split(','):
    if(c.strip()):
       columns.append(c.strip())

columns.append('target')
print(columns)
print(len(columns))

In [ ]:
attacks_types = {
    'normal': 'normal',
    'back': 'dos',
    'buffer_overflow': 'u2r',
    'ftp_write': 'r2l',
    'guess_passwd': 'r2l',
    'imap': 'r2l',
    'ipsweep': 'probe',
    'land': 'dos',
    'loadmodule': 'u2r',
    'multihop': 'r2l',
    'neptune': 'dos',
    'nmap': 'probe',
    'perl': 'u2r',
    'phf': 'r2l',
    'pod': 'dos',
    'portsweep': 'probe',
    'rootkit': 'u2r',
    'satan': 'probe',
    'smurf': 'dos',
    'spy': 'r2l',
    'teardrop': 'dos',
    'warezclient': 'r2l',
    'warezmaster': 'r2l',
}


path = "kddcup.data_10_percent.csv"
df = pd.read_csv(path,names=columns)

#Adding Attack Type column
df['Attack Type'] = df.target.apply(lambda r:attacks_types[r[:-1]])

df.head()

df['Attack Type'].value_counts()

# Visualizing the data

In [ ]:
def bar_graph(feature):
    df[feature].value_counts().plot(kind="bar",color="red")

In [ ]:
df.shape

In [ ]:
bar_graph('protocol_type')

In [ ]:
plt.figure(figsize=(15,3))
bar_graph('service')

In [ ]:
bar_graph('flag')

In [ ]:
bar_graph('logged_in')

In [ ]:
bar_graph('target')

In [ ]:
bar_graph('Attack Type')

In [ ]:
df.columns

        # Removing highly correlated columns

*   List item
*   List item



In [ ]:
df = df.dropna(axis = 'columns') # drop columns with NaN

df = df[[col for col in df if df[col].nunique() > 1]] # keep columns where there are more than 1 unique values

# corr = df.corr()
# Convert categorical variables into numerical representations using one-hot encoding
df_encoded = pd.get_dummies(df)

# Calculate the correlation matrix
corr_matrix = df_encoded.corr()

plt.figure(figsize=(15,12))

sns.heatmap(corr_matrix)

plt.show()

In [ ]:
df.head(1)

In [ ]:
print(df.columns)

In [ ]:
#This variable is highly correlated with num_compromised and should be ignored for analysis.
#(Correlation = 0.9938277978738366)
df.drop('num_root',axis = 1,inplace = True)

#This variable is highly correlated with serror_rate and should be ignored for analysis.
#(Correlation = 0.9983615072725952)
df.drop('srv_serror_rate',axis = 1,inplace = True)

#This variable is highly correlated with rerror_rate and should be ignored for analysis.
#(Correlation = 0.9947309539817937)
df.drop('srv_rerror_rate',axis = 1, inplace=True)

#This variable is highly correlated with srv_serror_rate and should be ignored for analysis.
#(Correlation = 0.9993041091850098)
df.drop('dst_host_srv_serror_rate',axis = 1, inplace=True)

#This variable is highly correlated with rerror_rate and should be ignored for analysis.
#(Correlation = 0.9869947924956001)
df.drop('dst_host_serror_rate',axis = 1, inplace=True)

#This variable is highly correlated with srv_rerror_rate and should be ignored for analysis.
#(Correlation = 0.9821663427308375)
df.drop('dst_host_rerror_rate',axis = 1, inplace=True)

#This variable is highly correlated with rerror_rate and should be ignored for analysis.
#(Correlation = 0.9851995540751249)
df.drop('dst_host_srv_rerror_rate',axis = 1, inplace=True)

#This variable is highly correlated with dst_host_srv_count and should be ignored for analysis.
#(Correlation = 0.9736854572953938)
df.drop('dst_host_same_srv_rate',axis = 1, inplace=True)

In [ ]:
df.head()

In [ ]:
df.columns

In [ ]:
df['Attack Type'].unique()

In [ ]:
df['target'].unique()

# Label encoding the features

In [ ]:
validateDf=df.copy()

In [ ]:
validateDf.to_csv('validate.csv')

In [ ]:
#protocol_type feature mapping
pmap = {'icmp':0,'tcp':1,'udp':2}
df['protocol_type'] = df['protocol_type'].map(pmap)

In [ ]:
#flag feature mapping
fmap = {'SF':0,'S0':1,'REJ':2,'RSTR':3,'RSTO':4,'SH':5 ,'S1':6 ,'S2':7,'RSTOS0':8,'S3':9 ,'OTH':10}
df['flag'] = df['flag'].map(fmap)

In [ ]:
#attack type feature mapping
amap = {'dos':0,'normal':1,'probe':2,'r2l':3,'u2r':4}
df['Attack Type'] = df['Attack Type'].map(amap)

In [ ]:
df.drop('service',axis = 1,inplace= True)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score

In [ ]:
import tensorflow as tf
from keras.models import Sequential, Model
from keras.layers import Dense, Conv1D, MaxPooling1D, Flatten, Dropout, Input, Concatenate, Add

In [ ]:
df = df.drop(['target',], axis=1)
print(df.shape)

# Target variable and train set
Y = df[['Attack Type']]
X = df.drop(['Attack Type',], axis=1)

sc = MinMaxScaler()
X = sc.fit_transform(X)

# Split test and train data
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.33,
                                                    random_state=42)
print(X_train.shape, X_test.shape)
print(Y_train.shape, Y_test.shape)

In [ ]:
# df.to_csv("ids.csv", index=False)

In [ ]:
# pd.read_csv("ids.csv")

## Shallow Neural Network

In [ ]:
shallow_model = Sequential([
    Dense(1024, input_dim=30, activation='relu'),
    Dropout(0.01),
    Dense(5, activation='softmax')
])

In [ ]:
shallow_model.compile(loss ='sparse_categorical_crossentropy', optimizer = 'adam', metrics = ['accuracy'])

In [ ]:
# tf.keras.utils.plot_model(shallow_model, to_file="shallow_model.png", show_shapes=True, show_layer_names=True)

In [ ]:
df.shape

In [ ]:
shallow_model.fit(X_train, Y_train.values.ravel(), epochs=10, batch_size=32)

## Deep Neural Network

In [ ]:
deep_model = Sequential([
    Dense(1024, input_dim=30, activation='relu'),
    Dropout(0.01),
    Dense(768, activation='relu'),
    Dropout(0.01),
    Dense(512, activation='relu'),
    Dropout(0.01),
    Dense(256, activation='relu'),
    Dropout(0.01),
    Dense(128, activation='relu'),
    Dropout(0.01),
    Dense(5, activation='softmax')
])

In [ ]:
deep_model.compile(loss ='sparse_categorical_crossentropy', optimizer = 'adam', metrics = ['accuracy'])

In [ ]:
# tf.keras.utils.plot_model(deep_model, to_file="deep_model.png", show_shapes=True)

In [ ]:
deep_model.fit(X_train, Y_train.values.ravel(), epochs=10, batch_size=32)

## Convolutional Neural Network

In [ ]:
cnn_model = Sequential([
    Conv1D(64, 3, padding="same", activation="relu", input_shape=(30,1)),
    MaxPooling1D(pool_size=(2)),
    Flatten(),
    Dense(128, activation="relu"),
    Dropout(0.5),
    Dense(5, activation="softmax")
])

inputs = Input(shape=(30, 1))
y = Conv1D(62, 3, padding="same", activation="relu", input_shape=(30,1))(inputs)
y = MaxPooling1D(pool_size=(2))(y)
y1 = Flatten()(y)

y = Dropout(0.5)(y)
y = Conv1D(62, 3, padding="same", activation="relu", input_shape=(30,1))(inputs)
y = MaxPooling1D(pool_size=(2))(y)
y2 = Flatten()(y)

y = Dropout(0.5)(y)
y = Conv1D(124, 3, padding="same", activation="relu", input_shape=(30,1))(inputs)
y = MaxPooling1D(pool_size=(2))(y)
y = Flatten()(y)
y = Dropout(0.5)(y)
y = Dense(256, activation="relu")(y)
y = Dropout(0.5)(y)
y = Dense(5, activation='softmax')(y)

y = Concatenate()([y, y1, y2])

outputs = Dense(5, activation='softmax')(y)
cnn_model = Model(inputs=inputs, outputs=outputs)

In [ ]:
cnn_model.compile(loss ='sparse_categorical_crossentropy', optimizer = 'adam', metrics = ['accuracy'])

In [ ]:
tf.keras.utils.plot_model(cnn_model, to_file="cnn_model.png", show_shapes=True)

In [ ]:
cnn_model.fit(X_train.reshape((-1,30,1)), Y_train.values.ravel(), epochs=10, batch_size=32)

# Testing the neural network

In [ ]:
shallow_preds_train = shallow_model.predict(X_train)
shallow_test = shallow_model.predict(X_test)

In [ ]:
deep_preds_train = deep_model.predict(X_train)
deep_test = deep_model.predict(X_test)

In [ ]:
cnn_preds_train = cnn_model.predict(X_train.reshape((-1,30,1)))
cnn_test = cnn_model.predict(X_test.reshape((-1,30,1)))

In [ ]:
X_train.shape

SHALLOW NEURAL NETWORK
Training Accuracy: 0.9993474201949281
Testing Accuracy: 0.9991167107288976

In [ ]:
print("SHALLOW NEURAL NETWORK")
print("Training Accuracy:", accuracy_score(Y_train, np.argmax(shallow_preds_train, axis=1)))
print("Testing Accuracy:", accuracy_score(Y_test, np.argmax(shallow_test, axis=1)))

DEEP NEURAL NETWORK
Training Accuracy: 0.9991419784044424
Testing Accuracy: 0.9990431032896392

In [ ]:
print("DEEP NEURAL NETWORK")
print("Training Accuracy:", accuracy_score(Y_train, np.argmax(deep_preds_train, axis=1)))
print("Testing Accuracy:", accuracy_score(Y_test, np.argmax(deep_test, axis=1)))

CONVOLUTIONAL NEURAL NETWORK
Training Accuracy: 0.9988730913551302
Testing Accuracy: 0.9986198605139026

In [ ]:
print("CONVOLUTIONAL NEURAL NETWORK")
print("Training Accuracy:", accuracy_score(Y_train, np.argmax(cnn_preds_train, axis=1)))
print("Testing Accuracy:", accuracy_score(Y_test, np.argmax(cnn_test, axis=1)))

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

# Function to print metrics for train and test sets
def print_metrics(model_name, y_true_train, y_pred_train, y_true_test, y_pred_test):
    print(f"\nMetrics for {model_name} Model:")

    # Train set
    print("\nTrain Set:")
    print("Accuracy:", accuracy_score(y_true_train, y_pred_train))
    print("Precision:", precision_score(y_true_train, y_pred_train, average='weighted'))
    print("Recall:", recall_score(y_true_train, y_pred_train, average='weighted'))
    print("F1-Score:", f1_score(y_true_train, y_pred_train, average='weighted'))
    print("Classification Report:\n", classification_report(y_true_train, y_pred_train))

    # Test set
    print("\nTest Set:")
    print("Accuracy:", accuracy_score(y_true_test, y_pred_test))
    print("Precision:", precision_score(y_true_test, y_pred_test, average='weighted'))
    print("Recall:", recall_score(y_true_test, y_pred_test, average='weighted'))
    print("F1-Score:", f1_score(y_true_test, y_pred_test, average='weighted'))
    print("Classification Report:\n", classification_report(y_true_test, y_pred_test))

# Predictions from the models
shallow_preds_train_labels = np.argmax(shallow_preds_train, axis=1)
shallow_test_labels = np.argmax(shallow_test, axis=1)

deep_preds_train_labels = np.argmax(deep_preds_train, axis=1)
deep_test_labels = np.argmax(deep_test, axis=1)

cnn_preds_train_labels = np.argmax(cnn_preds_train, axis=1)
cnn_test_labels = np.argmax(cnn_test, axis=1)

# Print metrics for each model
print_metrics("Shallow Neural Network", Y_train, shallow_preds_train_labels, Y_test, shallow_test_labels)
print_metrics("Deep Neural Network", Y_train, deep_preds_train_labels, Y_test, deep_test_labels)
print_metrics("Convolutional Neural Network", Y_train, cnn_preds_train_labels, Y_test, cnn_test_labels)


# More info

In [ ]:
df.columns

![image.png](https://i.ibb.co/QH93r9g/Screenshot-from-2022-11-03-02-32-15.png)

![image.png](https://i.ibb.co/B3J7kSV/Screenshot-from-2022-11-03-02-24-30.png)


Table 4: Summarizes attack types and 4 different categories: (1)DoS (Denial of Service attacks), (2) R2L (Root to Local attacks), (3) U2R (User to Root attack), (4) Probe (Probing attacks).

In [ ]:
# import pandas as pd
# import numpy as np
# from sklearn.preprocessing import MinMaxScaler

def preprocess_data(input_data, pmap, fmap, features, sc):
    """Preprocesses the input data."""
    input_df = pd.DataFrame([input_data])
    input_df['protocol_type'] = input_df['protocol_type'].map(pmap)
    input_df['flag'] = input_df['flag'].map(fmap)
    input_df = input_df[features]
    input_data_scaled = sc.transform(input_df)
    return input_data_scaled.reshape((-1,30,1)) # Reshape for CNN
def predict_attack(preprocessed_data, cnn_model):
    """Predicts the attack type using the CNN model."""
    prediction = cnn_model.predict(preprocessed_data)
    amap = {0: 'dos', 1: 'normal', 2: 'probe', 3: 'r2l', 4: 'u2r'}
    predicted_attack_type = amap[np.argmax(prediction)]
    return predicted_attack_type

def network_attack_pipeline(input_data, cnn_model, pmap, fmap, features, sc):
    """
    A pipeline for predicting network attacks.

    Args:
        input_data (dict): Input data dictionary.
        cnn_model (keras.Model): Trained CNN model.
        pmap (dict): Mapping for protocol_type.
        fmap (dict): Mapping for flag.
        features (list): List of features used in training.
        sc (MinMaxScaler): Scaler object.

    Returns:
        str: Predicted attack type.
    """
    preprocessed_data = preprocess_data(input_data, pmap, fmap, features, sc)
    predicted_attack = predict_attack(preprocessed_data, cnn_model)
    return predicted_attack

# Define your mappings, features, and scaler# You should use the same scaler object used during training

# Example usage:


In [ ]:
input=[0, 1,0,181,5450,0,0,0,0,0,1,0,0,0,0,0,0,0,8,8,0,0,1,0,0,9,9,0,0.11,0]

In [ ]:
# df.head(1).iloc[0,:].values.tolist()

In [ ]:
validateDf = validateDf.drop(['target',], axis=1)
validateDf.drop('service',axis = 1,inplace= True)

In [ ]:
len(input)

In [ ]:
amap = {'dos':0,'normal':1,'probe':2,'r2l':3,'u2r':4}
fmap = {'SF':0,'S0':1,'REJ':2,'RSTR':3,'RSTO':4,'SH':5 ,'S1':6 ,'S2':7,'RSTOS0':8,'S3':9 ,'OTH':10}
pmap = {'icmp':0,'tcp':1,'udp':2}

In [ ]:
pmap = {'icmp': 0, 'tcp': 1, 'udp': 2}
fmap = {'SF': 0, 'S0': 1, 'REJ': 2, 'RSTR': 3, 'RSTO': 4, 'SH': 5, 'S1': 6, 'S2': 7, 'RSTOS0': 8, 'S3': 9, 'OTH': 10}
features = ['duration', 'protocol_type', 'flag', 'src_bytes', 'dst_bytes', 'land',
       'wrong_fragment', 'urgent', 'hot', 'num_failed_logins', 'logged_in',
       'num_compromised', 'root_shell', 'su_attempted', 'num_file_creations',
       'num_shells', 'num_access_files', 'is_guest_login', 'count',
       'srv_count', 'serror_rate', 'rerror_rate', 'same_srv_rate',
       'diff_srv_rate', 'srv_diff_host_rate', 'dst_host_count',
       'dst_host_srv_count', 'dst_host_diff_srv_rate',
       'dst_host_same_src_port_rate', 'dst_host_srv_diff_host_rate']

In [ ]:
# network_attack_pipeline(input,cnn_model, pmap, fmap, features, sc)

In [ ]:
validateDf.head(1)

In [ ]:
# x.shape

In [ ]:
x=validateDf.iloc[177278,:-1]
validateDf.iloc[177278,-1]

In [ ]:
validateDf.sample(5)

In [ ]:
len(features)

In [ ]:
import joblib
import pickle

In [ ]:
cnn_model.save("cnn_model.h5")  # Save in HDF5 format

In [ ]:
joblib.dump(sc, "scaler.pkl")